# 🚀 Qwen3-VL-8B + LoRA | RTX 5060 Ti 16GB + Windows
**Qwen3-VL-8B-Instruct + QLoRA Fine-tuning for VQA**

> ⚠️ **설치 순서 중요**: PyTorch → transformers(git) → peft/accelerate 순으로 설치
> ⚠️ **VRAM OOM 시**: `MODEL_ID`를 `Qwen/Qwen2.5-VL-7B-Instruct`로 변경


In [3]:
# Cell 1-B: PyTorch 설치 (CUDA 12.1 - Colab 기본)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121


In [2]:
# Cell 1-C: transformers 최신 (Qwen3-VL 필수: >=4.57.0, git 소스 설치)
!pip install git+https://github.com/huggingface/transformers.git

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-atwed834
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-atwed834
  Resolved https://github.com/huggingface/transformers.git to commit edaac7db98e34208209fd67d8c66781b8c2e4a53
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [1]:
# Cell 1-D: 나머지 패키지 (transformers 설치 후에!)
!pip install "accelerate>=1.0.0" "peft>=0.14.0" "bitsandbytes>=0.45.0" -q

In [2]:
# Cell 1-E: 버전 호환성 수정 (gradio 호환)
!pip uninstall -y pillow pandas -q
!pip install "pillow<12,>=8.0" "pandas==2.2.2" -q

In [4]:
import pandas as pd

print(pd.__version__)

2.2.2


In [21]:
# 1. 기존 패키지 제거 및 베이스 환경 고정
#!pip uninstall -y torch torchvision torchaudio flash-attn
#!pip install -q --index-url https://download.pytorch.org/whl/cu124 torch torchvision torchaudio

# 2. H100(sm_90) 호환 Flash Attention 2 프리빌드 파일 설치 (30초 내외)
# 아래 링크는 공식 Dao-AILab에서 제공하는 최신 안정화 버전(v2.7.0) 예시입니다.
# 환경에 맞춰 가장 범용적인 cu124 버전을 설치합니다.
!pip install https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.0.post2/flash_attn-2.7.0.post2+cu124torch2.4cxx11abiTrue-cp312-cp312-linux_x86_64.whl#!pip install -q git+https://github.com/huggingface/transformers accelerate
#!pip install -q "peft>=0.13.2" datasets "pillow<12.0" "pandas==2.2.2"

  ERROR: HTTP error 404 while getting https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.0.post2/flash_attn-2.7.0.post2+cu124torch2.4cxx11abiTrue-cp312-cp312-linux_x86_64.whl#!pip
ERROR: Could not install requirement flash-attn==2.7.0.post2+cu124torch2.4cxx11abiTrue from https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.0.post2/flash_attn-2.7.0.post2+cu124torch2.4cxx11abiTrue-cp312-cp312-linux_x86_64.whl#!pip because of HTTP error 404 Client Error: Not Found for url: https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.0.post2/flash_attn-2.7.0.post2+cu124torch2.4cxx11abiTrue-cp312-cp312-linux_x86_64.whl for URL https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.0.post2/flash_attn-2.7.0.post2+cu124torch2.4cxx11abiTrue-cp312-cp312-linux_x86_64.whl#!pip


In [13]:
import zipfile
import os

zip_path = '/content/2026-ssafy-ai-15-2.zip'
extract_path = '/content/'

# 저장할 폴더가 없다면 생성
if not os.path.exists(extract_path):
    os.makedirs(extract_path)

# 압축 해제
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f"압축 해제 완료: {extract_path}")

압축 해제 완료: /content/


In [14]:
# Cell 2: 버전 및 GPU 확인
import torch, transformers, peft, accelerate, bitsandbytes
print(f"PyTorch:        {torch.__version__}")
print(f"transformers:   {transformers.__version__}")
print(f"peft:           {peft.__version__}")
print(f"accelerate:     {accelerate.__version__}")
print(f"bitsandbytes:   {bitsandbytes.__version__}")
print(f"CUDA:           {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name()}")
    print(f"VRAM:           {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

PyTorch:        2.10.0+cu128
transformers:   5.5.0.dev0
peft:           0.18.1
accelerate:     1.13.0
bitsandbytes:   0.49.2
CUDA:           True
GPU:            NVIDIA H100 80GB HBM3


AttributeError: 'torch._C._CudaDeviceProperties' object has no attribute 'total_mem'

In [5]:
# Cell 3: 임포트 + CUDA 최적화 (Colab H100)
import os, re, math, random, gc
import pandas as pd
import numpy as np
from PIL import Image, ImageOps, ImageEnhance
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Dict, List, Any
from transformers import (
    AutoModelForImageTextToText,
    AutoProcessor,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm.auto import tqdm
import torchvision.transforms as transforms

Image.MAX_IMAGE_PIXELS = None

# ★ Colab CUDA 최적화
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True  # ★ Colab: True (안정성 더 이상 문제 없음)

# Flash Attention 2 활성화
try:
    from flash_attn import flash_attn_func
    print("✓ Flash Attention 2 감지됨")
except ImportError:
    print("⚠ Flash Attention 2 미설치 - 자동 설치 권장: !pip install flash-attn")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

⚠ Flash Attention 2 미설치 - 자동 설치 권장: !pip install flash-attn
Device: cuda


In [6]:
# Cell 4: 하이퍼파라미터 (개선됨)
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

IMAGE_SIZE = 1024
MAX_NEW_TOKENS = 4
SEED = 42

NUM_EPOCHS = 5              # ↑ 3→5 (더 많은 학습)
BATCH_SIZE = 8
GRAD_ACCUM = 2         # effective batch = 8
LEARNING_RATE = 5e-5
WARMUP_RATIO = 0.1
PATIENCE = 2            # Early Stopping 인내심

INFERENCE_BATCH = 4
NUM_WORKERS = 4

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

train_df = pd.read_csv("train.csv")
test_df  = pd.read_csv("test.csv")
print(f"학습: {len(train_df)}개, 테스트: {len(test_df)}개")

학습: 5073개, 테스트: 5074개


In [7]:
# Cell 5: 모델 로드 + LoRA (개선됨)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels= 256 * 256,
    max_pixels= 1024 * 1024,
    trust_remote_code=True,
)

base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",   # ★ Windows: sdpa 안정적
)

base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=32,                  # ↑ 16→32 (표현력 증가)
    lora_alpha=64,         # ↑ 32→64 (스케일 조정)
    lora_dropout=0.1,      # ↑ 0.05→0.1 (정규화 강화)
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

if torch.cuda.is_available():
    print(f"VRAM 사용: {torch.cuda.memory_allocated()/1024**3:.1f} GB / 16.0 GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

trainable params: 87,293,952 || all params: 8,854,417,648 || trainable%: 0.9859
VRAM 사용: 8.6 GB / 16.0 GB


In [8]:
# Cell 6: 프롬프트 (개선됨)
SYSTEM_INSTRUCT = (
    "You are an expert visual question answering system specialized in multiple-choice questions. "
    "Your task is to analyze the image carefully, understand the question, and select the BEST answer. "
    "CRITICAL RULES:\n"
    "1. Examine the image for all relevant details and context\n"
    "2. Consider spatial relationships and visual elements\n"
    "3. Eliminate obviously wrong options first\n"
    "4. Respond with ONLY a single lowercase letter: a, b, c, or d\n"
    "5. NO explanations, punctuation, or extra characters\n"
    "EXAMPLES: a | b | c | d"
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f"Question: {question}\n\n"
        f"Options:\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "Based on the image, the correct answer is:"
    )

In [9]:
# Cell 7: 데이터셋 + 콜레이터 (데이터 증강 추가)
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train
        self.current_epoch = 0

    def set_epoch(self, epoch):
        """epoch 정보 설정 (증강 확률 제어용)"""
        self.current_epoch = epoch

    def __len__(self):
        return len(self.df)

    def apply_augmentation(self, img):
        """이미지 증강 적용 (epoch마다 다름)"""
        if not self.train:
            return img

        # epoch 진행에 따라 증강 강도 증가
        aug_prob = min(0.3 + self.current_epoch * 0.1, 0.9)  # epoch 0: 30%, epoch 1: 40%, ...

        if random.random() < aug_prob:
            # 회전 (±15도)
            if random.random() < 0.3:
                angle = random.uniform(-15, 15)
                img = img.rotate(angle, expand=False, fillcolor='white')

            # 색상 변환 (밝기, 대비, 채도)
            if random.random() < 0.3:
                brightness_enhancer = ImageEnhance.Brightness(img)
                img = brightness_enhancer.enhance(random.uniform(0.8, 1.2))

            if random.random() < 0.3:
                contrast_enhancer = ImageEnhance.Contrast(img)
                img = contrast_enhancer.enhance(random.uniform(0.8, 1.2))

            if random.random() < 0.3:
                color_enhancer = ImageEnhance.Color(img)
                img = color_enhancer.enhance(random.uniform(0.8, 1.2))

            # 좌우 뒤집기 (20%)
            if random.random() < 0.2:
                img = ImageOps.mirror(img)

        return img

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")

        # 데이터 증강 적용
        img = self.apply_augmentation(img)

        q, a, b, c, d = str(row["question"]), str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        user_text = build_mc_prompt(q, a, b, c, d)

        messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[
                {"type":"image","image":img},
                {"type":"text","text":user_text}
            ]}
        ]
        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append({"role":"assistant","content":[{"type":"text","text":gold}]})

        return {"messages": messages, "image": img}


@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images, prompt_texts = [], [], []
        for sample in batch:
            msgs, img = sample["messages"], sample["image"]

            full_text = self.processor.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=False
            )
            texts.append(full_text)
            images.append(img)

            if self.train:
                prompt_text = self.processor.apply_chat_template(
                    msgs[:-1], tokenize=False, add_generation_prompt=True
                )
                prompt_texts.append(prompt_text)

        enc = self.processor(
            text=texts, images=images,
            padding=True, return_tensors="pt"
        )

        if self.train:
            labels = enc["input_ids"].clone()
            for idx, pt in enumerate(prompt_texts):
                prompt_enc = self.processor.tokenizer(
                    pt, return_tensors="pt", add_special_tokens=False
                )
                prompt_len = prompt_enc["input_ids"].shape[1]
                labels[idx, :prompt_len] = -100
            if self.processor.tokenizer.pad_token_id is not None:
                labels[labels == self.processor.tokenizer.pad_token_id] = -100
            enc["labels"] = labels

        return enc

In [10]:
# Cell 8: 데이터 분리 + DataLoader (Colab 최적화)
train_df_shuffled = train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
split = int(len(train_df_shuffled) * 0.9)
train_subset = train_df_shuffled.iloc[:split]
valid_subset = train_df_shuffled.iloc[split:]
print(f"학습: {len(train_subset)}개, 검증: {len(valid_subset)}개")

train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)

# ★ Colab(Linux): num_workers=4, persistent_workers=True
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=DataCollator(processor, True),
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True,
)
valid_loader = DataLoader(
    valid_ds, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=DataCollator(processor, True),
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True,
)

학습: 4565개, 검증: 508개


In [11]:
# Cell 9: 학습 루프 (데이터 증강 + Early Stopping)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
num_training_steps = NUM_EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
num_warmup_steps = int(num_training_steps * WARMUP_RATIO)

scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps)
scaler = torch.amp.GradScaler("cuda", enabled=True)

best_val_loss = float("inf")
patience_counter = 0
SAVE_DIR = "qwen3_vl_8b_lora_best"

print(f"총 학습 스텝: {num_training_steps}, 워밍업: {num_warmup_steps}")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print("모델 준비 완료!\n")

global_step = 0
for epoch in range(NUM_EPOCHS):
    # ★ epoch마다 데이터 증강 설정
    train_ds.set_epoch(epoch)

    model.train()
    running, epoch_loss, epoch_steps = 0.0, 0.0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [train]", unit="batch")

    for step, batch in enumerate(pbar, start=1):
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            loss = model(**batch).loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running += loss.item()

        if step % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

    # ★ Early Stopping (검증 루프)
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in tqdm(valid_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [valid]", unit="batch"):
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                loss = model(**batch).loss
            val_loss += loss.item()

    val_loss /= len(valid_loader)
    train_loss = running / (step * GRAD_ACCUM)
    print(f"Epoch {epoch+1} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        model.save_pretrained(SAVE_DIR)
        print(f"✓ 모델 저장! (Val Loss 개선: {val_loss:.4f})")
    else:
        patience_counter += 1
        print(f"⚠ 성능 미개선 ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print(f"🛑 Early Stopping! (Epoch {epoch+1}에서 중단)")
            break

    print()

총 학습 스텝: 1430, 워밍업: 143
Effective batch size: 16
모델 준비 완료!



Epoch 1/5 [train]:   0%|          | 0/571 [00:00<?, ?batch/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch 1/5 [valid]:   0%|          | 0/64 [00:00<?, ?batch/s]

Epoch 1 - Train Loss: 2.2306, Val Loss: 6.8715
✓ 모델 저장! (Val Loss 개선: 6.8715)



...

Epoch 4/5 [valid]:   0%|          | 0/64 [00:00<?, ?batch/s]

Epoch 4 - Train Loss: 1.7175, Val Loss: 6.8646
✓ 모델 저장! (Val Loss 개선: 6.8646)



Epoch 5/5 [train]:   0%|          | 0/571 [00:00<?, ?batch/s]

Epoch 5/5 [valid]:   0%|          | 0/64 [00:00<?, ?batch/s]

Epoch 5 - Train Loss: 1.7170, Val Loss: 6.8645
✓ 모델 저장! (Val Loss 개선: 6.8645)



In [12]:
# Cell 10: 추론 (최적화됨 - Beam Search 제거)
def extract_choice(text: str) -> str:
    text = text.strip().lower()
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return "a"
    last = lines[-1]
    if last in ["a","b","c","d"]:
        return last
    for pat in [r'\b([abcd])\b', r'\(([abcd])\)', r'answer\s*(?:is|:)\s*([abcd])']:
        m = re.search(pat, text)
        if m:
            return m.group(1)
    return "a"

processor.tokenizer.padding_side = 'left'

def batch_inference(model, processor, test_df, batch_size=2):
    model.eval()
    preds = []
    for start in tqdm(range(0, len(test_df), batch_size), desc="Inference", unit="batch"):
        end = min(start + batch_size, len(test_df))
        rows = test_df.iloc[start:end]
        texts, images = [], []

        for _, row in rows.iterrows():
            img = Image.open(row["path"]).convert("RGB")
            user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])
            msgs = [
                {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
                {"role":"user","content":[
                    {"type":"image","image":img},
                    {"type":"text","text":user_text}
                ]}
            ]
            text = processor.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True
            )
            texts.append(text)
            images.append(img)

        inputs = processor(
            text=texts, images=images,
            padding=True, return_tensors="pt"
        ).to(device)

        with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
            out_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,           # ✓ 결정적 (argmax)
                temperature=0.0,           # ✓ 1글자 선택에 최적
                eos_token_id=processor.tokenizer.eos_token_id,
            )

        # 생성된 토큰만 추출
        input_len = inputs["input_ids"].shape[1]
        for i_b in range(len(rows)):
            gen_ids = out_ids[i_b][input_len:]
            txt = processor.decode(gen_ids, skip_special_tokens=True)
            preds.append(extract_choice(txt))

        del inputs, out_ids
        torch.cuda.empty_cache()

    return preds


preds = batch_inference(model, processor, test_df, batch_size=INFERENCE_BATCH)

os.makedirs("content", exist_ok=True)
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("content/submission.csv", index=False)
print(f"\nSaved content/submission.csv ({len(submission)}개)")
print(submission["answer"].value_counts().sort_index())

Inference:   0%|          | 0/1269 [00:00<?, ?batch/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Saved content/submission.csv (5074개)
answer
a    1302
b    1273
c    1277
d    1222
Name: count, dtype: int64


In [ ]:
from google.colab import drive
drive.mount('/content/drive')